# Complete governed P&ID proposal

This example turns a simulated separator, compressor, cooler and pump into a governed P&ID proposal. It exports DEXPI and sidecars, independently imports the exchange file with PyDEXPI, and renders the generated instrumentation and safeguarding. Generated objects remain review-required and are not fit for construction.

In [ ]:
import json, os, shutil, sys
from pathlib import Path
ROOT = Path(os.environ.get('NEQSIM_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'pom.xml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT, recompile=False, verbose=False))
J = ns.JClass
output = ROOT / 'build' / 'complete-pid-design-synthesis'
shutil.rmtree(output, ignore_errors=True); output.mkdir(parents=True)
print('Output directory:', output)

In [ ]:
fluid = J('neqsim.thermo.system.SystemSrkEos')(298.15, 60.0)
fluid.addComponent('methane', 0.90); fluid.addComponent('n-heptane', 0.10)
feed = J('neqsim.process.equipment.stream.Stream')('20-FEED-001', fluid)
separator = J('neqsim.process.equipment.separator.Separator')('20-VG-001', feed)
compressor = J('neqsim.process.equipment.compressor.Compressor')('20-KA-001', separator.getGasOutStream())
cooler = J('neqsim.process.equipment.heatexchanger.Cooler')('20-HA-001', compressor.getOutletStream())
pump = J('neqsim.process.equipment.pump.Pump')('20-PA-001', separator.getLiquidOutStream())
process = J('neqsim.process.processmodel.ProcessSystem')()
for item in (feed, separator, compressor, cooler, pump): process.add(item)
project = J('neqsim.process.engineering.NorsokOffshoreEngineeringBuilder').from_('Complete P&ID proposal', process).projectId('PID-NOTEBOOK').registerProposedInstruments(True).build()
ProtectedItem = J('neqsim.process.safety.overpressure.ProtectedItem')
ReliefScenario = J('neqsim.process.safety.overpressure.ReliefScenario')
cause = J('neqsim.process.safety.overpressure.ReliefCause').BLOCKED_OUTLET
phase = J('neqsim.process.safety.overpressure.ReliefPhase').VAPOUR
scenario = ReliefScenario.Builder('Blocked outlet', cause).phase(phase).reliefRateKgPerS(3.0).reliefTemperatureK(330.0).molarMassKgPerMol(0.020).compressibility(0.95).specificHeatRatio(1.25).build()
study = J('neqsim.process.safety.overpressure.OverpressureProtectionStudy')(ProtectedItem('20-VG-001', 70.0).setReliefSetPressureBara(68.0)).addScenario(scenario)
project.addOverpressureStudy(study)
print('Process equipment:', process.getUnitOperations().size())

In [ ]:
basis = J('neqsim.process.engineering.pid.PidDesignBasis')('NORSOK-COMPLETE-PID-PROPOSALS', '20')
catalog = J('neqsim.process.engineering.pid.NorsokPidRuleCatalog').completeProposals()
model = J('neqsim.process.engineering.pid.PidDesignSynthesizer').synthesize(project, basis, catalog)
report = J('neqsim.process.engineering.pid.PidCompletenessValidator').validate(model)
Paths = J('java.nio.file.Paths')
exported = J('neqsim.process.engineering.pid.PidEngineeringPackageExporter').export(project, model, Paths.get(str(output)))
counts = {}
for element in model.getElements(): counts[str(element.getType())] = counts.get(str(element.getType()), 0) + 1
print('Generated proposals:', model.getElements().size(), counts)
print('Structurally complete:', report.isStructurallyComplete(), '| ready for approval:', report.isReadyForApproval())
print('Sidecars:', exported.getDesignModelFile(), exported.getCompletenessReportFile())

In [ ]:
from pydexpi.loaders import ProteusSerializer
from pydexpi.loaders.svg_loader import DrawDiagram
from IPython.display import SVG, display
model_from_dexpi = ProteusSerializer().load(str(output), 'plant-pydexpi.xml')
assert model_from_dexpi.diagram is not None
DrawDiagram(model_from_dexpi.diagram, padding=5.0, pretty=True).save_svg('complete-pid-pydexpi', str(output))
svg = output / 'complete-pid-pydexpi.svg'
render_report = {'status': 'IMPORT_AND_RENDER_PASSED', 'source': 'plant-pydexpi.xml', 'rendered': svg.name, 'fitnessForConstruction': False}
(output / 'pydexpi-render-report.json').write_text(json.dumps(render_report, indent=2) + '\n', encoding='utf-8')
display(SVG(filename=str(svg)))
print(render_report)